# 04 — MNIST 손글씨 분류: MLP와 CNN 비교

**목표:** 같은 이미지 데이터로 두 신경망을 직접 학습시키고 정확도·학습 시간·GPU 메모리를 비교한다.

MNIST에는 손글씨 숫자(0~9) 이미지가 70,000장 있다. 학습용 60,000장 중 55,000장으로 학습하고 5,000장으로 검증한다. 별도 테스트 10,000장은 마지막에만 사용한다.

RunPod의 GPU·PyTorch·Jupyter 환경은 이미 갖춰졌다고 가정한다. torchvision과 matplotlib도 설치되어 있어야 한다. MNIST를 처음 다운로드할 때만 인터넷이 필요하다. 각 셀을 순서대로 실행하라.

In [ ]:
import time
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

assert torch.cuda.is_available(), "GPU를 사용하는 Jupyter 커널을 선택하세요."
device = torch.device("cuda")
torch.manual_seed(42)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 1. 데이터 준비

`ToTensor`는 흑백 이미지 한 장을 shape `(1, 28, 28)`인 Tensor로 만든다. 픽셀 값은 0~1 범위다. `DataLoader`는 학습할 이미지들을 배치 단위로 제공한다.

이 실습은 학습용 55,000장, 검증용 5,000장, 테스트용 10,000장을 사용한다. 검증과 테스트는 가중치 갱신에 사용하지 않는다.

In [ ]:
batch_size = 128
epochs = 3
learning_rate = 0.001

data_dir = Path("data/mnist")
transform = transforms.ToTensor()
full_train = datasets.MNIST(root=data_dir, train=True, transform=transform, download=True)
test_data = datasets.MNIST(root=data_dir, train=False, transform=transform, download=True)
train_data, val_data = random_split(
    full_train, [55_000, 5_000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

images, labels = next(iter(train_loader))
print("train, val, test:", len(train_data), len(val_data), len(test_data))
print("이미지 배치 shape:", images.shape, "정답 shape:", labels.shape)

## 2. 이미지 실제로 보기

데이터에 들어 있는 숫자 이미지와 대응하는 정답 라벨을 확인한다.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, label in zip(axes.flat, images[:10], labels[:10]):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(f"label {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. 신경망 두 개 만들기

**MLP:** 이미지의 784개 픽셀을 하나의 벡터로 만든 다음 선형층과 ReLU를 거쳐 숫자 10개의 점수를 출력한다.

**CNN:** 이미지의 공간 배치를 유지한 상태에서 `Conv2d`와 `MaxPool2d`를 거쳐 숫자 10개의 점수를 출력한다.

두 모델의 출력은 확률이 아니라 클래스별 점수(logits)다. `CrossEntropyLoss`에 점수를 그대로 넣는다.

In [ ]:
def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 128),
        nn.ReLU(),
        nn.Linear(128, 10),
    )

def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 7 * 7, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    )

factories = {"MLP": make_mlp, "CNN": make_cnn}
for name, factory in factories.items():
    sample_model = factory()
    with torch.inference_mode():
        sample_output = sample_model(images[:4])
    print(name, "출력 shape:", sample_output.shape,
          "파라미터 수:", sum(p.numel() for p in sample_model.parameters()))

## 4. 검증 함수

`model(x)`가 이미지마다 숫자 10개의 점수를 반환하면 `argmax(dim=1)`으로 가장 큰 점수의 클래스 번호를 선택한다. `model.eval()`은 평가 모드이고, `inference_mode()`는 자동미분 관련 처리를 생략한다.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            scores = model(x)
            loss_sum += nn.functional.cross_entropy(scores, y, reduction="sum").item()
            correct += (scores.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total

## 5. Epoch × Batch 이중 반복으로 학습

내부 반복문에서는 배치 하나마다 `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`을 한 번씩 실행한다. 외부 반복문은 전체 학습용 데이터를 `epochs`번 사용한다.

각 Epoch 이후 검증 정확도를 확인한다. 학습시간에는 데이터 로딩과 CPU-GPU 전송도 포함된다. PyTorch의 GPU 메모리 피크는 드라이버 전체 VRAM 사용량과 다르다.

In [ ]:
history = {}
trained_models = {}

for name, factory in factories.items():
    torch.manual_seed(42)
    model = factory().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    records = []
    torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(epochs):
        model.train()
        total_loss, seen = 0.0, 0
        torch.cuda.synchronize()
        started = time.perf_counter()

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y_batch)
            seen += len(y_batch)

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        val_loss, val_accuracy = evaluate(model, val_loader)
        records.append({
            "train_loss": total_loss / seen,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "train_seconds": elapsed,
        })
        print(
            f"{name} | epoch {epoch + 1}/{epochs} | "
            f"train_loss {total_loss / seen:.4f} | "
            f"val_loss {val_loss:.4f} | "
            f"val_acc {val_accuracy:.2%} | "
            f"train_time {elapsed:.1f}s"
        )

    peak_mib = torch.cuda.max_memory_allocated(device) / 2**20
    print(f"{name} PyTorch GPU peak allocated: {peak_mib:.1f} MiB")
    history[name] = {"epochs": records, "peak_mib": peak_mib}
    trained_models[name] = model.cpu()
    del model, optimizer
    torch.cuda.empty_cache()

## 6. 학습 결과 비교

마지막 Epoch의 검증 정확도, 전체 학습시간, PyTorch VRAM 피크를 각각 확인한다. 두 모델은 같은 데이터와 학습률·배치 크기·Epoch를 사용했지만 파라미터 수와 연산량은 같지 않다.

In [ ]:
for name, result in history.items():
    rows = result["epochs"]
    print(
        f"{name}: val_accuracy {rows[-1]['val_accuracy']:.2%}, "
        f"training_time {sum(r['train_seconds'] for r in rows):.1f}s, "
        f"peak {result['peak_mib']:.1f} MiB"
    )

plt.figure(figsize=(7, 4))
for name, result in history.items():
    values = [r["val_accuracy"] for r in result["epochs"]]
    plt.plot(range(1, epochs + 1), values, marker="o", label=name)
plt.xticks(range(1, epochs + 1))
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 7. 모델을 선택한 뒤 테스트 10,000장으로 평가

검증 정확도가 높은 모델을 선택한다. 테스트 데이터는 모델을 선택하거나 학습시키는 데 사용하지 않았으므로 마지막에 한 번 평가한다.

In [ ]:
best_name = max(history, key=lambda n: history[n]["epochs"][-1]["val_accuracy"])
best_model = trained_models[best_name].to(device)
test_loss, test_accuracy = evaluate(best_model, test_loader)
print(f"선택한 모델: {best_name}")
print(f"테스트 loss {test_loss:.4f} | 정확도 {test_accuracy:.2%}")

## 8. 실제 이미지에 대한 예측

테스트 이미지 12장을 출력하고 정답과 예측값을 나란히 비교한다. 마지막에는 숫자 하나에 대한 logits와 Softmax 값도 출력한다.

In [ ]:
best_model.eval()
test_images, test_targets = next(iter(test_loader))
with torch.inference_mode():
    scores = best_model(test_images.to(device))
    predictions = scores.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].squeeze(0), cmap="gray")
    ax.set_title(f"actual {test_targets[i].item()} / pred {predictions[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("첫 이미지 logits:", scores[0].cpu().tolist())
print("첫 이미지 Softmax:", scores[0].softmax(dim=0).cpu().tolist())

## 직접 바꿔볼 것

- `batch_size` 32, 128, 512의 학습시간·VRAM·검증 정확도 비교
- `epochs` 1, 3, 5의 손실 및 검증 정확도 비교
- MLP 은닉층 크기 32, 128, 256 비교
- CNN `Conv2d` 출력 채널 수 변경 (다음 층의 입력 채널 수와 마지막 Linear의 입력 크기도 일치시킨다)

MNIST는 `data/mnist`에 저장되며 Git에 업로드하지 않는다. 테스트 정확도를 보고 반복해서 모델을 고르면 테스트 데이터를 모델 선택에 사용한 것이 되므로 주의한다.